In [18]:
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.datasets import imdb
from keras.preprocessing import sequence
from keras.models import load_model


In [19]:
# Loading the dataset
word_index = imdb.get_word_index()
reverse_word_index = {value:key for key, value in word_index.items()}


In [20]:
# Loading the model
model = load_model("simple_rnn_imdb.keras")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,939,077 (15.03 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,626,052 (10.02 MB)

In [21]:
# Loading the weights just for inspection 
# At time I also save the architecture and the weights separately and then combine the two
model.get_weights()

[array([[-0.47761887,  0.38338277, -1.3972616 , ..., -0.4443223 ,
         -0.693501  ,  1.010346  ],
        [ 0.08133309, -0.07015155, -0.07539808, ...,  0.03513094,
          0.07302278,  0.02562043],
        [ 0.07291416, -0.03069489, -0.02270865, ...,  0.02668995,
          0.05298709,  0.04877202],
        ...,
        [ 0.13793428,  0.15541399,  0.02555833, ..., -0.04227943,
          0.10926551,  0.16739896],
        [-0.01569088,  0.02033051,  0.10087852, ..., -0.00218341,
         -0.0520928 , -0.05393869],
        [ 0.21004604,  0.2215173 , -0.04766076, ..., -0.08996867,
          0.2587339 ,  0.17588177]], dtype=float32),
 array([[ 0.03739758, -0.12549293,  0.00650068, ..., -0.14579692,
         -0.04826736, -0.15835224],
        [ 0.03674784, -0.1190308 ,  0.04531922, ...,  0.16467783,
         -0.01602861, -0.00959803],
        [ 0.01984419,  0.03048494, -0.12591098, ...,  0.10981216,
         -0.12951428,  0.11453044],
        ...,
        [ 0.05382712, -0.00744809, -0.1

In [22]:
# Adding Helper functions
# 1) Decoding reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i-3,'?') for i in encoded_review])

# 2) Preprocessing the user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word,2)+ 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [23]:
# Adding prediction function
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input)
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    return sentiment, prediction[0][0]

In [27]:
# Adding user input 
# Checking prediction
sample_reviews = [
                  'Movie was very mysterious. We kept wondering what is going to happen next.',
                  ]

for i in sample_reviews :
    sentiment,score = predict_sentiment(i)

    print(f"Review : {i}")
    print(f"Sentiment : {sentiment}")
    print(f"Score : {score}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
Review : Movie was very mysterious. We kept wondering what is going to happen next.
Sentiment : Positive
Score : 0.6754970550537109
